In [1]:
import os
import sys
import psutil

print("Python:", sys.executable)
print("Python version:", sys.version)
print("CPU cores:", os.cpu_count())
print("RAM:", round(psutil.virtual_memory().total / 1024**3, 1), "GB")

BASE_DIR = "/home/ubuntu/student_resource/student_resource"
PIPELINE = "/home/ubuntu/ER_final_pipeline_optimized.py"
OUT_DIR = "/home/ubuntu/output"

print("\nBASE_DIR exists:", os.path.exists(BASE_DIR))
print("Pipeline exists:", os.path.exists(PIPELINE))
print("Output directory:", os.path.exists(OUT_DIR))

Python: /home/ubuntu/venv/bin/python
Python version: 3.14.4 (main, Aug 20 2026, 10:41:58) [GCC 15.2.0]
CPU cores: 8
RAM: 61.8 GB

BASE_DIR exists: True
Pipeline exists: True
Output directory: True


In [2]:
from pathlib import Path

base = Path("/home/ubuntu/student_resource/student_resource")

files = [
    base / "dataset/train/train_ground_truth.tsv",
    base / "dataset/train/train_source1.tsv",
    base / "dataset/train/train_source2.tsv",
    base / "dataset/train/train_source3.tsv",
    base / "dataset/test/test_source1.tsv",
    base / "dataset/test/test_source2.tsv",
    base / "dataset/test/test_source3.tsv",
    base / "utils/validate_submission.py",
]

for f in files:
    print("OK" if f.exists() else "MISSING", "->", f)

OK -> /home/ubuntu/student_resource/student_resource/dataset/train/train_ground_truth.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/train/train_source1.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/train/train_source2.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/train/train_source3.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/test/test_source1.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/test/test_source2.tsv
OK -> /home/ubuntu/student_resource/student_resource/dataset/test/test_source3.tsv
OK -> /home/ubuntu/student_resource/student_resource/utils/validate_submission.py


In [3]:
import re

pipeline_path = "/home/ubuntu/ER_final_pipeline_optimized.py"

with open(pipeline_path, "r", encoding="utf-8") as f:
    code = f.read()

for name in [
    "BASE_DIR",
    "OUT_DIR",
    "DEV_MODE",
    "N_JOBS",
    "Q_CHUNK",
    "N_TREES",
    "RESUME",
    "ONLY_COUNTRIES",
    "THRESH",
]:
    m = re.search(rf"^{name}\s*=.*$", code, re.MULTILINE)
    print(m.group(0) if m else f"{name}: NOT FOUND")

BASE_DIR  = "/home/ubuntu/student_resource/student_resource"
OUT_DIR   = "/home/ubuntu/output"
DEV_MODE = False
N_JOBS       = os.cpu_count()
Q_CHUNK   = 100            # S1 rows per blocking task (bounds memory of the sparse product)
N_TREES   = 300
RESUME         = False
ONLY_COUNTRIES = None     # None = all countries; e.g. ["France"] to run only some (other countries then get empty rows - a trial, not a submission)
THRESH    = None           # None = tune on the held-out fold; or force a value e.g. 0.8


In [4]:
import subprocess
from pathlib import Path

PIPELINE = "/home/ubuntu/ER_final_pipeline_optimized.py"
LOG = "/home/ubuntu/optimized_run.log"
OUT = "/home/ubuntu/output"

Path(OUT).mkdir(parents=True, exist_ok=True)

cmd = [
    "/home/ubuntu/venv/bin/python",
    PIPELINE
]

with open(LOG, "a", buffering=1) as log:
    process = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd="/home/ubuntu",
        start_new_session=True
    )

print("Pipeline started!")
print("PID:", process.pid)
print("Log:", LOG)
print("Output directory:", OUT)

Pipeline started!
PID: 5402
Log: /home/ubuntu/optimized_run.log
Output directory: /home/ubuntu/output


In [5]:
import subprocess

result = subprocess.run(
    ["pgrep", "-af", "ER_final_pipeline_optimized.py"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "Pipeline process not found.")

5402 /home/ubuntu/venv/bin/python /home/ubuntu/ER_final_pipeline_optimized.py



In [6]:
from pathlib import Path

log = Path("/home/ubuntu/optimized_run.log")

if log.exists():
    lines = log.read_text(errors="replace").splitlines()
    print("\n".join(lines[-40:]))
else:
    print("Log file not found.")

DEV_MODE = False | N_PER = 2500 | N_JOBS = 8
[   0.2 min | 1.6 GB peak] train strata=76  sampled S1=148509
[   0.2 min | 4.2 GB peak] train pool=694870 records
[   0.5 min | 4.2 GB peak]    India: normalised 40186 S1 + 211823 pool records in 16s
[   0.6 min | 4.2 GB peak]    matrices built (retrieval cap=2500); retrieving 40186 x 211823 ...
[   0.6 min | 4.2 GB peak]       pmap 41/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 81/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 121/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 161/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 201/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 242/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 282/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 322/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 362/402 tasks done
[   0.6 min | 4.2 GB peak]       pmap 402/402 tasks done
[   0.7 min | 4.2 GB peak]    re-ranked 2273907 retrieved pairs -> kept 478217
[   0.7 mi

In [7]:
import psutil

m = psutil.virtual_memory()

print(f"RAM total:     {m.total / 1024**3:.1f} GB")
print(f"RAM used:      {m.used / 1024**3:.1f} GB")
print(f"RAM available: {m.available / 1024**3:.1f} GB")
print(f"RAM usage:     {m.percent}%")

RAM total:     61.8 GB
RAM used:      3.2 GB
RAM available: 58.6 GB
RAM usage:     5.2%
